# using IODevices under PythonNET

compile IODevices with the same target (64 vs 32 bits) as your python interpreter,
but making sure there will be no forms displayed by changing these default values: 
showdevicesatstartup=false and showmessages=false (although the latter can be set in python too).

Install the clr/pythonnet package in your Python distribution

https://pythonnet.github.io/pythonnet/python.html


In [3]:
#initialization

import clr  #pythonnet  or clr package 

IOdll_path = r'IODevices4py.dll'  #version compiled with showdevicesatstartup=false, showmessages=false, otherwise python will crash

clr.AddReference(IOdll_path)

# for better error messages you can replace clr.AddReference(IOdll_path) by:
#clr.AddReference("System")
#import System
#System.Reflection.Assembly.LoadFile(IOdll_path)   #like clr.AddReference but gives more detailed info in case of error

import IODevices as io     #import namespace



In [3]:
#now define a device: here an Agilent DMM

Ag34410_addr = "USB0::2391::1543::MY45001884::INSTR"
dev1=io.VisaDevice("ag", Ag34410_addr);
dev1.showmessages=False   #disable message window if not already done in the dll

#set some parameters if needed
dev1.enablepoll=True


In [9]:
# test of blocking query

# for passing ref/out args see https://stackoverflow.com/questions/54692267/python-net-call-c-sharp-method-which-has-a-return-value-and-an-out-parameter
status,resp=dev1.QueryBlocking("Read?", "", False)    #dummy string as second parameter to select the right version,

print ("status: ", status, ", response: ", resp)


status:  0 , response:  -7.10378968E-05


In [8]:
# test of async query

#see https://stackoverflow.com/questions/22384783/how-to-pass-python-callback-to-c-sharp-function-call


# first define a callback function: must match the IOCallback definition, i.e. take exactly one parameter, no return value,
# and then interpret it according to the IOQuery definition (otherwise the kernel may crash)
def cbdev1(q):             
    print ("cbdev1 called with command=", q.cmd, " status=", q.status," response=", q.ResponseAsString, " received at", q.timeend) 
    print ("response time:", q.timeend.Subtract(q.timestart).TotalSeconds) 

    
# the syntax to instantiate a NET delegate corresponding to our callback function : io.IODevice.IOCallback(cbdev1)
# this delegate can then be passed on to the dll async functions
status=dev1.QueryAsync("Read?", io.IODevice.IOCallback(cbdev1), False)  

print ("queryasync status=", status)



queryasync status= 0
cbdev1 called with command= Read?  status= 0  response= -1.74548056E-04  received at 15/07/2022 19:05:21
response time: 0.0640376


In [ ]:
dev1.Dispose() # here dev1 does not go out of scope (when finalizer is called) so you have to explicitly call Dispose 
                # to release hardware resources (for example for SerialDevice it will release the COM port etc.)  
